<a href="https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks and baseline rule

#### Signal 1 — Content age / staleness

**Verdict: MIXED**

The bucket check does not show a consistent relationship between older content and decline. Pages aged 91–180 days had the highest decline rate at about 61.1%, while the 181–365 day group was lower at about 46.7%. The 366+ day group had a 60.0% decline rate, but it contains only 5 pages.

Therefore, staleness is useful as a supporting signal, but it should not be treated as a strong standalone indicator.

#### Signal 2 — Search visibility

**Verdict: CONFIRMED**

The bucket check shows that pages with meaningful search visibility generally have higher declining rates than pages with very low impressions. The decline rate increases from about 38.9% for pages with 0–100 impressions to more than 60% in the higher-impression buckets.

Therefore, search visibility is a useful signal for prioritizing pages that have enough search activity to justify review.

#### My baseline rule

I will rank pages higher when they have meaningful search visibility and show some evidence of staleness.

The score will give more weight to recent search visibility and a smaller supporting weight to content age. This keeps the baseline focused on pages where there is enough observable search activity to justify human review, while avoiding the assumption that older content is automatically a problem.

Each ranked page will receive one reason code and one action label.

The rule is a decision-support baseline, not a guarantee that refreshing a page will improve its performance.

In [8]:
import pandas as pd

# Load the starter dataset
url = "https://raw.githubusercontent.com/Sharmadipti/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Total pages: {len(df):,}")

# ---------------------------------------------------------
# Signal 1: Staleness / days since last update
# ---------------------------------------------------------
print("\nSIGNAL 1 — STALENESS: days_since_last_update")

staleness_buckets = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["0-90 days", "91-180 days", "181-365 days", "366+ days"]
)

staleness_table = (
    df.assign(staleness_bucket=staleness_buckets)
      .groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_pages=("trend_direction", lambda x: (x == "down").sum())
      )
      .reset_index()
)

staleness_table["declining_rate"] = (
    staleness_table["declining_pages"] / staleness_table["n"]
)

print(staleness_table)
print(f"n checked: {staleness_table['n'].sum():,}")

stale_flag_count = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
).sum()

print("\nPages meeting the staleness flag threshold:")
print(f"{stale_flag_count:,}")

# ---------------------------------------------------------
# Signal 2: Search visibility / impressions
# ---------------------------------------------------------
print("\nSIGNAL 2 — SEARCH VISIBILITY: impressions_90d")

visibility_buckets = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=[
        "0-100",
        "101-500",
        "501-1,000",
        "1,001-5,000",
        "5,001+"
    ]
)

visibility_table = (
    df.assign(visibility_bucket=visibility_buckets)
      .groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_pages=("trend_direction", lambda x: (x == "down").sum())
      )
      .reset_index()
)

visibility_table["declining_rate"] = (
    visibility_table["declining_pages"] / visibility_table["n"]
)

print(visibility_table)
print(f"n checked: {visibility_table['n'].sum():,}")

visibility_flag_count = (df["impressions_90d"] >= 500).sum()

print("\nPages meeting the visibility threshold:")
print(f"{visibility_flag_count:,}")

Total pages: 30,000

SIGNAL 1 — STALENESS: days_since_last_update
  staleness_bucket      n  declining_pages  declining_rate
0        0-90 days  20655            10576        0.512031
1      91-180 days   9171             5604        0.611057
2     181-365 days    169               79        0.467456
3        366+ days      5                3        0.600000
n checked: 30,000

Pages meeting the staleness flag threshold:
17

SIGNAL 2 — SEARCH VISIBILITY: impressions_90d
  visibility_bucket     n  declining_pages  declining_rate
0             0-100  8006             3116        0.389208
1           101-500  5279             3190        0.604281
2         501-1,000  3206             1925        0.600437
3       1,001-5,000  7359             4669        0.634461
4            5,001+  6150             3362        0.546667
n checked: 30,000

Pages meeting the visibility threshold:
16,726


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
import os
import pandas as pd

# Load the starter dataset
url = "https://raw.githubusercontent.com/Sharmadipti/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# ---------------------------------------------------------
# Baseline scoring rule
# ---------------------------------------------------------

# Search visibility points
df["visibility_points"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=[0, 1, 2, 3, 4]
).astype(int)

# Staleness points
# 0-90 days   -> 0 points
# 91-180 days -> 1 point
# 181-365 days -> 1 point
# 366+ days   -> 2 points
df["staleness_points"] = 0

df.loc[
    df["days_since_last_update"].between(91, 365),
    "staleness_points"
] = 1

df.loc[
    df["days_since_last_update"] >= 366,
    "staleness_points"
] = 2

# Final baseline score
df["baseline_score"] = (
    df["visibility_points"] * 2
    + df["staleness_points"]
)

# ---------------------------------------------------------
# Reason code
# ---------------------------------------------------------

df["reason_code"] = "VISIBILITY"

df.loc[
    df["staleness_points"] > df["visibility_points"],
    "reason_code"
] = "STALENESS"

df.loc[
    (df["staleness_points"] > 0)
    & (df["visibility_points"] > 0)
    & (df["staleness_points"] == df["visibility_points"]),
    "reason_code"
] = "VISIBILITY_AND_STALENESS"

# ---------------------------------------------------------
# Action label
# ---------------------------------------------------------

df["action"] = "Monitor"

df.loc[
    df["baseline_score"] >= 6,
    "action"
] = "Review"

# ---------------------------------------------------------
# Rank all pages
# ---------------------------------------------------------

queue = (
    df[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "action",
            "impressions_90d",
            "days_since_last_update"
        ]
    ]
    .sort_values(
        ["baseline_score", "impressions_90d", "days_since_last_update"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "days_since_last_update"
    ]
]

# ---------------------------------------------------------
# Write the required CSV
# ---------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"Ranked queue created: {len(queue):,} pages")
print(f"Saved to: {output_path}")

print("\nTop 20:")
queue.head(20)

Ranked queue created: 30,000 pages
Saved to: work/outputs/baseline_action_score.csv

Top 20:


,rank,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update
0,1,content_5fe46e04994d,client_4e07408562,9,VISIBILITY,Review,517715,104
1,2,content_2dba2b1f9536,client_6208ef0f77,9,VISIBILITY,Review,443434,104
2,3,content_2c2606c5d176,client_19581e27de,9,VISIBILITY,Review,347399,104
3,4,content_cb112fce36be,client_19581e27de,9,VISIBILITY,Review,309910,104
4,5,content_9532f197bbc8,client_4e07408562,9,VISIBILITY,Review,309192,104
5,6,content_36ff89c8214e,client_19581e27de,9,VISIBILITY,Review,295097,104
6,7,content_b28d1efd668f,client_6208ef0f77,9,VISIBILITY,Review,286608,104
7,8,content_813e88069237,client_6208ef0f77,9,VISIBILITY,Review,233561,104
8,9,content_c21024970297,client_19581e27de,9,VISIBILITY,Review,211366,104
9,10,content_c8e9d6ab9013,client_19581e27de,9,VISIBILITY,Review,208678,104


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the top 20 ranked pages

top20 = queue.head(20).copy()

print("TOP-20 REVIEW")
print("=" * 80)

for _, row in top20.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"{row['content_id']} | "
        f"Action: {row['action']} | "
        f"Why: {row['impressions_90d']:,} impressions give this page "
        f"strong search visibility and place it high in the baseline queue. | "
        f"What would make it wrong: the page may already be performing well, "
        f"so high visibility alone may not justify a refresh."
    )

TOP-20 REVIEW
Rank 1: content_5fe46e04994d | Action: Review | Why: 517,715 impressions give this page strong search visibility and place it high in the baseline queue. | What would make it wrong: the page may already be performing well, so high visibility alone may not justify a refresh.
Rank 2: content_2dba2b1f9536 | Action: Review | Why: 443,434 impressions give this page strong search visibility and place it high in the baseline queue. | What would make it wrong: the page may already be performing well, so high visibility alone may not justify a refresh.
Rank 3: content_2c2606c5d176 | Action: Review | Why: 347,399 impressions give this page strong search visibility and place it high in the baseline queue. | What would make it wrong: the page may already be performing well, so high visibility alone may not justify a refresh.
Rank 4: content_cb112fce36be | Action: Review | Why: 309,910 impressions give this page strong search visibility and place it high in the baseline queue. | What 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Weak picks + leakage check
# ---------------------------------------------------------

print("WEAK PICKS — BOTTOM 10")
print("=" * 80)

weak_picks = queue.tail(10).sort_values("rank")

for _, row in weak_picks.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"{row['content_id']} | "
        f"Score: {int(row['baseline_score'])} | "
        f"Action: {row['action']} | "
        f"Impressions: {int(row['impressions_90d']):,} | "
        f"Days since update: {int(row['days_since_last_update'])}"
    )

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

print("\nLEAKAGE CHECK")
print("=" * 80)

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "went_dark"
]

score_features = [
    "impressions_90d",
    "days_since_last_update"
]

leakage_found = [
    feature for feature in forbidden_features
    if feature in score_features
]

print("Features used by the baseline score:")
print(score_features)

print("\nForbidden label-derived/future features checked:")
print(forbidden_features)

if leakage_found:
    print("\nWARNING: Leakage detected!")
    print(leakage_found)
else:
    print("\nPASS: No label-derived or future-window features are used.")

print("\nBaseline CSV exists:")
print(os.path.exists("work/outputs/baseline_action_score.csv"))

WEAK PICKS — BOTTOM 10
Rank 29991: content_bb600f317035 | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29992: content_92ceb4aee549 | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29993: content_994b0a4e4dde | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29994: content_5168e96834b9 | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29995: content_b5fb35404aed | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29996: content_8bce3371c63c | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29997: content_2a843f006d86 | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29998: content_1d9eca1ce9cd | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 29999: content_9ffe1e2e3575 | Score: 0 | Action: Monitor | Impressions: 1 | Days since update: 1
Rank 30000: content_0a22a2eeefdd | Score: 0 | Action: Monit

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.